In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import plotly.express as px
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

In [2]:
df = pd.read_csv("../data/raw/pollution_sample.csv", parse_dates=['Date Local'])

### Explore Data

In [3]:
df.head()

,station,State,City,Date Local,Unnamed: 0,State Code,County Code,Site Num,NO2 Mean,NO2 1st Max Value,...,O3 AQI,SO2 Mean,SO2 1st Max Value,SO2 1st Max Hour,SO2 AQI,CO Mean,CO 1st Max Value,CO 1st Max Hour,CO AQI,Year
0,11_1_41,District Of Columbia,Washington,2010-01-01,48351.5,11.0,1.0,41.0,19.208333,30.0,...,24.0,2.862500,4.80,13.5,7.0,0.962939,1.25,5.5,14.0,2010.0
1,11_1_41,District Of Columbia,Washington,2010-01-02,48355.5,11.0,1.0,41.0,8.708333,10.0,...,29.0,3.204167,4.00,7.0,6.0,0.772917,0.80,6.5,9.0,2010.0
2,11_1_41,District Of Columbia,Washington,2010-01-03,48359.5,11.0,1.0,41.0,7.291667,10.0,...,29.0,3.033333,3.65,13.5,6.0,0.827083,0.90,17.5,10.0,2010.0
3,11_1_41,District Of Columbia,Washington,2010-01-04,48363.5,11.0,1.0,41.0,13.041667,22.0,...,21.0,4.441667,5.80,22.5,9.0,0.906250,0.95,3.5,10.0,2010.0
4,11_1_41,District Of Columbia,Washington,2010-01-05,48367.5,11.0,1.0,41.0,15.173913,22.0,...,22.0,5.531987,7.30,5.0,11.0,0.925947,1.00,14.5,11.0,2010.0


In [4]:
df.shape

(70922, 25)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70922 entries, 0 to 70921
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   station            70922 non-null  object        
 1   State              70922 non-null  object        
 2   City               70922 non-null  object        
 3   Date Local         70922 non-null  datetime64[ns]
 4   Unnamed: 0         70922 non-null  float64       
 5   State Code         70922 non-null  float64       
 6   County Code        70922 non-null  float64       
 7   Site Num           70922 non-null  float64       
 8   NO2 Mean           70922 non-null  float64       
 9   NO2 1st Max Value  70922 non-null  float64       
 10  NO2 1st Max Hour   70922 non-null  float64       
 11  NO2 AQI            70922 non-null  float64       
 12  O3 Mean            70922 non-null  float64       
 13  O3 1st Max Value   70922 non-null  float64       
 14  O3 1st

In [6]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Date Local,70922,2013-01-24 21:00:03.959279104,2010-01-01 00:00:00,2011-09-04 00:00:00,2013-01-24 00:00:00,2014-06-23 00:00:00,2015-12-31 00:00:00,NaN
Unnamed: 0,70922.0,71224.816044,1.5,53132.0,74365.5,93813.5,134573.5,32740.373427
State Code,70922.0,26.492823,4.0,12.0,24.0,40.0,56.0,15.133508
County Code,70922.0,60.606441,1.0,13.0,33.0,113.0,209.0,58.431886
Site Num,70922.0,788.477863,2.0,10.0,41.0,1004.0,9997.0,1719.895174
NO2 Mean,70922.0,11.305964,-2.0,5.440227,9.5,15.375,77.247826,7.921427
NO2 1st Max Value,70922.0,23.57464,-2.0,12.4,22.0,33.1,156.9,13.872214
NO2 1st Max Hour,70922.0,11.917007,0.0,5.0,9.0,20.0,23.0,8.077365
NO2 AQI,70922.0,22.035758,0.0,11.0,21.0,31.0,111.0,13.151682
O3 Mean,70922.0,0.026384,0.0,0.018375,0.026083,0.033958,0.069875,0.010995


In [7]:
df.isnull().sum()

station               0
State                 0
City                  0
Date Local            0
Unnamed: 0            0
State Code            0
County Code           0
Site Num              0
NO2 Mean              0
NO2 1st Max Value     0
NO2 1st Max Hour      0
NO2 AQI               0
O3 Mean               0
O3 1st Max Value      0
O3 1st Max Hour       0
O3 AQI                0
SO2 Mean              0
SO2 1st Max Value     0
SO2 1st Max Hour      0
SO2 AQI               0
CO Mean               0
CO 1st Max Value      0
CO 1st Max Hour       0
CO AQI               43
Year                  0
dtype: int64

In [8]:
df.duplicated().sum()

0

###